In [5]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Num GPUs Available:  1


In [6]:
import numpy as np                # For numerical operations
import pandas as pd               # For handling structured data
import os                         # For file and directory management
import glob                       # For pattern matching in file paths
from PIL import Image             # For image processing
import cv2                        # For image and video processing
import matplotlib.pyplot as plt   # For visualizing images and results
import seaborn as sns             # For creating statistical plots
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split  # For splitting datasets
from sklearn.preprocessing import LabelEncoder        # For encoding labels
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report
from PIL import Image


from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam

# Load pre-trained ResNet50 without the top layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)  # Output for binary classification (real/fake)

# Combine into final model
model = Model(inputs=base_model.input, outputs=predictions)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Compile the model
model.compile(optimizer=Adam(lr=0.0001), loss='binary_crossentropy', metrics=['accuracy'])




C:\ProgramData\anaconda3\envs\py310\lib\site-packages\keras\optimizers\optimizer_v2\adam.py:114: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D

# Suppress the DecompressionBombWarning
Image.MAX_IMAGE_PIXELS = None
# Allow loading of truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# Load pre-trained ResNet50 without the top layer
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Add custom classification layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)  # Output for binary classification (real/fake)

# Combine into final model
model = Model(inputs=base_model.input, outputs=predictions)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Compile the model
model.compile(optimizer=Adam(lr=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# Prepare data generators
train_dir = 'dataset/train'
test_dir = 'dataset/test'

train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary')

validation_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary')

# Train the model
model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=10)

# Unfreeze the base model layers for fine-tuning (optional)
for layer in base_model.layers:
    layer.trainable = True

model.compile(optimizer=Adam(lr=1e-5), loss='binary_crossentropy', metrics=['accuracy'])

model.fit(
    train_generator,
    steps_per_epoch=train_generator.samples // train_generator.batch_size,
    validation_data=validation_generator,
    validation_steps=validation_generator.samples // validation_generator.batch_size,
    epochs=10)

# Evaluate the model
validation_loss, validation_accuracy = model.evaluate(validation_generator)
print(f'Validation Accuracy: {validation_accuracy}')



Found 29606 images belonging to 2 classes.
Found 12000 images belonging to 3 classes.
Epoch 1/10
925/925 [==============================] - 662s 713ms/step - loss: 0.4515 - accuracy: 0.8110 - val_loss: 0.7556 - val_accuracy: 0.5548
Epoch 2/10
925/925 [==============================] - 611s 661ms/step - loss: 0.4340 - accuracy: 0.8195 - val_loss: 0.6961 - val_accuracy: 0.6060
Epoch 3/10
925/925 [==============================] - 606s 655ms/step - loss: 0.4277 - accuracy: 0.8214 - val_loss: 0.8847 - val_accuracy: 0.5254
Epoch 4/10
925/925 [==============================] - 661s 714ms/step - loss: 0.4224 - accuracy: 0.8253 - val_loss: 0.7763 - val_accuracy: 0.5846
Epoch 5/10
925/925 [==============================] - 644s 696ms/step - loss: 0.4196 - accuracy: 0.8267 - val_loss: 0.7515 - val_accuracy: 0.6039
Epoch 6/10
925/925 [==============================] - 635s 686ms/step - loss: 0.4169 - accuracy: 0.8267 - val_loss: 0.7353 - val_accuracy: 0.6104
Epoch 7/10
925/925 [==================

In [8]:
model.save('my_model_new.h5')  # Saves in the SavedModel format


In [1]:
# # Function to make predictions
# def predict_image(img_path):
#     img = image.load_img(img_path, target_size=(224, 224))
#     img_array = image.img_to_array(img)
#     img_array = np.expand_dims(img_array, axis=0) / 255.0
#     prediction = model.predict(img_array)
#     if prediction < 0.5:
#         print(f'The image at {img_path} is predicted to be Fake.')
#     else:
#         print(f'The image at {img_path} is predicted to be Real.')

# # Example usage
# predict_image('dataset\test\real\0002.jpg')


In [ ]:
# from PIL import Image
# import os

# def resave_images(directory):
#     for root, _, files in os.walk(directory):
#         for file in files:
#             file_path = os.path.join(root, file)
#             try:
#                 img = Image.open(file_path)
#                 img = img.convert("RGB")  # Convert to RGB format
#                 img.save(file_path)  # Save the image, overwriting the original file
#                 print(f"Resaved: {file_path}")
#             except (IOError, SyntaxError) as e:
#                 print(f"Skipping problematic image: {file_path}, error: {e}")

# # Resave images in your test dataset
# resave_images("dataset/test")
# resave_images("dataset/train")


In [ ]:
# import numpy as np
# from tensorflow.keras.preprocessing.image import ImageDataGenerator
# from tensorflow.keras.applications.resnet50 import preprocess_input
# from tensorflow.keras.utils import Sequence
# from PIL import Image

# class SafeImageDataGenerator(Sequence):
#     def __init__(self, directory, datagen, batch_size, target_size, subset, class_mode, shuffle, seed):
#         self.directory = directory
#         self.datagen = datagen
#         self.batch_size = batch_size
#         self.target_size = target_size
#         self.subset = subset
#         self.class_mode = class_mode
#         self.shuffle = shuffle
#         self.seed = seed
#         self.generator = datagen.flow_from_directory(
#             directory,
#             target_size=target_size,
#             batch_size=batch_size,
#             class_mode=class_mode,
#             subset=subset,
#             shuffle=shuffle,
#             seed=seed
#         )

#     def __len__(self):
#         return len(self.generator)

#     def __getitem__(self, index):
#         batch_x, batch_y = self.generator[index]
#         valid_x, valid_y = [], []
#         for i in range(len(batch_x)):
#             try:
#                 img = Image.fromarray(np.uint8(batch_x[i] * 255))
#                 img.verify()  # Verify the integrity of the image
#                 valid_x.append(batch_x[i])
#                 valid_y.append(batch_y[i])
#             except (IOError, SyntaxError) as e:
#                 print(f"Skipping corrupted image in batch {index}, image {i}: {e}")
#         return np.array(valid_x), np.array(valid_y)

# # Create the SafeImageDataGenerator instances
# train_generator = SafeImageDataGenerator(
#     directory='dataset/train',
#     datagen=train_datagen,
#     batch_size=16,  # Reduced batch size
#     target_size=(224, 224),
#     subset='training',
#     class_mode='binary',
#     shuffle=True,
#     seed=42
# )

# validation_generator = SafeImageDataGenerator(
#     directory='dataset/train',
#     datagen=train_datagen,
#     batch_size=16,  # Reduced batch size
#     target_size=(224, 224),
#     subset='validation',
#     class_mode='binary',
#     shuffle=False,
#     seed=42
# )

# # Fit the model
# history = model.fit(
#     train_generator,
#     steps_per_epoch=len(train_generator),
#     epochs=1,  # You can adjust this
#     validation_data=validation_generator,
#     validation_steps=len(validation_generator)
# )


In [ ]:
# import os
# import numpy as np
# from PIL import Image
# from tensorflow.keras.preprocessing.image import img_to_array
# from tensorflow.keras.applications.resnet50 import preprocess_input
# from tensorflow.keras.models import load_model

# def load_images(directory, target_size):
#     images = []
#     labels = []
#     class_indices = {'real': 0, 'fake': 1}  # Update according to your class names

#     for class_name, class_idx in class_indices.items():
#         class_dir = os.path.join(directory, class_name)
#         if not os.path.exists(class_dir):
#             continue
#         for file in os.listdir(class_dir):
#             file_path = os.path.join(class_dir, file)
#             try:
#                 img = Image.open(file_path)
#                 img = img.convert("RGB")
#                 img = img.resize(target_size)
#                 img_array = img_to_array(img)
#                 img_array = preprocess_input(img_array)
#                 images.append(img_array)
#                 labels.append(class_idx)
#             except (IOError, SyntaxError) as e:
#                 print(f"Skipping problematic image: {file_path}, error: {e}")

#     return np.array(images), np.array(labels)

# # Load images from the test directory
# test_images, test_labels = load_images('dataset/test', target_size=(224, 224))

# # Load your trained model
# # model = load_model('path_to_your_trained_model.h5')

# # Make predictions on the test images
# predictions = model.predict(test_images)

# # Convert predictions to binary labels (0 or 1)
# predicted_labels = (predictions > 0.5).astype(int).flatten()

# # Evaluate the performance
# from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# accuracy = accuracy_score(test_labels, predicted_labels)
# report = classification_report(test_labels, predicted_labels)
# conf_matrix = confusion_matrix(test_labels, predicted_labels)

# print(f'Accuracy: {accuracy:.2f}')
# print('Classification Report:')
# print(report)
# print('Confusion Matrix:')
# print(conf_matrix)


In [ ]:
# import numpy as np
# from tensorflow.keras.preprocessing.image import load_img, img_to_array
# from tensorflow.keras.applications.resnet50 import preprocess_input
# from tensorflow.keras.models import load_model
# import os

# # Load your trained model
# # model = load_model('path_to_your_trained_model.h5')

# def predict_image(image_path, model):
#     # Load and preprocess the image
#     try:
#         img = load_img(image_path, target_size=(224, 224))
#         img_array = img_to_array(img)
#         img_array = preprocess_input(img_array)
#         img_array = np.expand_dims(img_array, axis=0)
        
#         # Make prediction
#         prediction = model.predict(img_array)
#         predicted_label = (prediction > 0.5).astype(int).flatten()[0]
        
#         # Return the result
#         if predicted_label == 0:
#             return "Real"
#         else:
#             return "Fake"
#     except (IOError, SyntaxError) as e:
#         return f"Error loading image: {e}"

# # Example usage
# image_path = 'dataset2/test/real/0010.jpg'  # Update with your image path
# result = predict_image(image_path, model)
# print(f"The image '{os.path.basename(image_path)}' is {result}.")
